# AquaSize AI -- Phase 1: Reef Fish Species Detection
### Run this notebook on Google Colab (free T4 GPU): Runtime > Change runtime type > GPU

This trains a species detector on catch-style images and benchmarks how
accuracy degrades under realistic catch-image conditions (lighting, angle,
blur, compression, clutter) -- NOT underwater turbidity, which is the wrong
domain for landed-catch photos.

Pipeline: dataset -> train YOLOv8-seg -> evaluate (accuracy, confusion
matrix, precision/recall) -> robustness benchmark across stress conditions
-> plots for your report/essay.


## 1. Setup

In [ ]:
!pip install -q ultralytics roboflow albumentations opencv-python-headless
import sys, os
sys.path.append('src')  # if you've uploaded the AquaSize_AI/src folder to this Colab session


## 2. Get a dataset

Search Roboflow Universe for a reef/fish species dataset (e.g. search
"fish species detection" or "reef fish"). Grab your free API key from
roboflow.com/settings, then fill in the workspace/project/version below.

If no Red Sea-specific dataset exists (likely true right now), start with
a general reef fish dataset for this proof of concept and note the
Red-Sea-taxa gap explicitly in your write-up -- that's an honest framing,
not a weakness to hide.


In [ ]:
from data_prep import download_roboflow_dataset

ROBOFLOW_API_KEY = "YOUR_KEY_HERE"
ds_path = download_roboflow_dataset(
    api_key=ROBOFLOW_API_KEY,
    workspace="your-workspace",
    project="your-fish-project",
    version=1,
)
print(ds_path)


## 3. Train (Phase 1: detection/segmentation + species ID)

In [ ]:
!python src/train.py --data {ds_path}/data.yaml --epochs 40 --model yolov8n-seg.pt \
    --project runs/aquasize --name phase1_species_detect


## 4. Evaluate: accuracy, confusion matrix, precision/recall

In [ ]:
import pandas as pd
from evaluate_metrics import classification_report_dict, confusion_matrix_df
import matplotlib.pyplot as plt
import seaborn as sns

# Build y_true / y_pred lists from your test set (see robustness_benchmark.py
# for the full inference loop pattern), then:
# report = classification_report_dict(y_true, y_pred)
# print("Accuracy:", report["overall_accuracy"])
# print(report["per_class"])
#
# cm = confusion_matrix_df(y_true, y_pred)
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
# plt.title("Species Confusion Matrix (baseline, no stress)")
# plt.tight_layout(); plt.savefig("docs/confusion_matrix.png", dpi=150)


## 5. Robustness benchmark across catch-image conditions

This is the deliverable that directly answers the brief's "test how
different image settings influence the performance of the model" ask.


In [ ]:
!python src/robustness_benchmark.py \
    --weights runs/aquasize/phase1_species_detect/weights/best.pt \
    --test-dir {ds_path}/test \
    --out docs/robustness_report.csv


In [ ]:
report = pd.read_csv("docs/robustness_report.csv")
plt.figure(figsize=(9,5))
sns.barplot(data=report, x="condition", y="accuracy", color="#2b7a78")
plt.xticks(rotation=35, ha="right")
plt.ylabel("Species-ID accuracy")
plt.title("Accuracy by catch-image condition")
plt.tight_layout()
plt.savefig("docs/robustness_by_condition.png", dpi=150)
plt.show()


## 6. Phase 2 (stretch goal): sizing + statistical validation

Only start this once Phase 1 above is solid and validated. See
`src/sizing.py` -- it converts a fish mask + in-frame reference object into
an estimated length, then validates against known lengths with MAE, RMSE,
R^2, and a Bland-Altman bias/limits-of-agreement report (the standard way
biometrics papers report measurement-method agreement).


In [ ]:
from sizing import compute_px_per_cm, estimate_length, validate_against_ground_truth, validation_report_df

# predicted_lengths, ground_truth_lengths = [...], [...]   # from a small hand-measured validation set
# stats_report = validate_against_ground_truth(predicted_lengths, ground_truth_lengths)
# print(stats_report)
# validation_report_df(predicted_lengths, ground_truth_lengths).to_csv("docs/sizing_validation.csv", index=False)
